# Kiduka Document Indexer (LCEL Version)
Use this notebook to upload agricultural knowledge and project documentation to the Pinecone vector store using modern LCEL.

In [ ]:
%pip install langchain_community

In [2]:
%pip uninstall pinecone-plugin-inference -y

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import sys
import time
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# Ensure the root directory is in the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from chatbot.utils.config import config

# Load Environment Variables
load_dotenv()

print("Environment and dependencies loaded.")

Environment and dependencies loaded.


### 1. Load and Index PDFs

In [2]:
pdf_folder_path = "./documents"
docs = []

if not os.path.exists(pdf_folder_path):
    os.makedirs(pdf_folder_path)

print(f"Loading PDFs from: {pdf_folder_path}")
for filename in os.listdir(pdf_folder_path):
    if filename.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder_path, filename))
        docs.extend(loader.load())

if docs:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)
    
    embeddings = OpenAIEmbeddings()
    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
    
    # Use the configured index name
    index_name = config.PINECONE_INDEX_NAME
    
    if index_name not in pc.list_indexes().names():
        print(f"Creating index {index_name}...")
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)
    
    print(f"Indexing {len(splits)} chunks into {index_name}...")
    vectorstore = PineconeVectorStore.from_documents(
        splits, 
        embeddings, 
        index_name=index_name
    )
    print("Indexing complete.")
else:
    print("No documents found to index.")

Loading PDFs from: ./documents


/opt/anaconda3/envs/kiduka-env/lib/python3.11/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Creating index kiduka-soil...
Indexing 16 chunks into kiduka-soil...
Indexing complete.


### 2. Test RAG Chain (LCEL)

In [3]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

embeddings = OpenAIEmbeddings()
vectorstore = PineconeVectorStore(index_name=config.PINECONE_INDEX_NAME, embedding=embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_template(config.RAG_TEMPLATE)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
output_parser = StrOutputParser()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

question = "How can I improve my soil pH for maize?"
print(f"Question: {question}")
print("Response:")
for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)

Question: How can I improve my soil pH for maize?
Response:
To improve your soil pH for maize, you can consider the following steps:

1. **Soil Testing**: First, conduct a soil test to determine the current pH level and understand the specific needs of your soil.

2. **Lime Application**: If your soil pH is too low (acidic), you can apply agricultural lime (calcium carbonate) to raise the pH. The amount needed will depend on your soil's current pH and the desired level.

3. **Organic Amendments**: Incorporate organic materials such as compost or well-rotted manure, which can help improve soil structure and pH over time.

4. **Avoid Acidic Fertilizers**: Be cautious with fertilizers that can further lower soil pH, such as ammonium-based fertilizers. Instead, opt for neutral or alkaline fertilizers.

5. **Regular Monitoring**: After making amendments, continue to monitor your soil pH regularly to ensure it remains within the optimal range for maize, which is typically between 6.0 and 7.0